# 01 · Alertas
Lê o relatório completo de alertas da plataforma SCCON (todas as classes) e
complementa com a planilha da CMF/Idaf. Grava `alertas.parquet` na camada prata.

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path

In [ ]:
# Caminhos do projeto (relativos à pasta notebooks/)
bronze_dir = Path("../dados/bronze")
silver_dir = Path("../dados/prata")
sccon_file = bronze_dir / "sccon_alertas_completo_2026-11-23.csv"
cmf_file = bronze_dir / "Controle de alertas - CMF-SEAMA.xlsx"

# Data de referência: última atualização da planilha da CMF (cabeçalho da aba DADOS)
header = pd.read_excel(cmf_file, sheet_name="DADOS", nrows=0).columns
reference_date = next(pd.Timestamp(c) for c in header if isinstance(c, datetime))
print("Planilha da CMF atualizada em", reference_date.date())

In [ ]:
# Relatório completo da plataforma: um alerta por linha, todas as classes
df_sccon = pd.read_csv(sccon_file, encoding="ISO-8859-1", dtype=str)
sccon_columns = {
    "idt_alerta": "id_alerta",
    "tipo": "tipo_alerta",
    "cidade": "municipio",
    "area_ha": "area_ha",
    "dat_antes": "data_imagem_antes",
    "dat_depois": "data_imagem_depois",
    "dat_public": "data_publicacao",
    "lat_centro": "latitude",
    "long_centr": "longitude",
}
df_alerts = df_sccon[list(sccon_columns)].rename(columns=sccon_columns)

# Da planilha da CMF: data de publicação antiga, município, unidade responsável e marcação de resposta
df_control = pd.read_excel(cmf_file, sheet_name="CONTROLE SCCON")
cmf_columns = {
    "idt_alerta": "id_alerta",
    "publicação": "data_publicacao_cmf",
    "cidade do alerta": "municipio_cmf",
    "Responsável": "unidade_responsavel",
    "resposta": "tem_resposta",
}
df_cmf = df_control[list(cmf_columns)].rename(columns=cmf_columns).dropna(subset=["id_alerta"])
df_cmf["id_alerta"] = df_cmf["id_alerta"].astype("int64")
df_cmf["tem_resposta"] = df_cmf["tem_resposta"].astype("string").str.strip().str.lower().eq("sim")

In [ ]:
# Converte coordenadas em graus, minutos e segundos (ex.: 18°53'9.2"S) para graus decimais
def dms_to_decimal(series):
    pattern = r"(?P<deg>\d+)\D+(?P<min>\d+)\D+(?P<sec>\d+(?:\.\d+)?)\D*(?P<hem>[NSEW])"
    parts = series.astype("string").str.extract(pattern)
    decimal = (parts["deg"].astype(float)
               + parts["min"].astype(float) / 60
               + parts["sec"].astype(float) / 3600)
    # Sul e Oeste são negativos
    return decimal.where(~parts["hem"].isin(["S", "W"]), -decimal)

In [ ]:
# Limpeza
df_alerts["id_alerta"] = df_alerts["id_alerta"].astype("int64")

# Separa município e UF do município principal (ex.: "Aimorés - MG")
city_parts = df_alerts["municipio"].str.extract(r"^(?P<nome>.+?)\s*-\s*(?P<uf>[A-Z]{2})$")
df_alerts["municipio"] = city_parts["nome"].str.strip()
df_alerts["uf"] = city_parts["uf"]

df_alerts["area_ha"] = pd.to_numeric(
    df_alerts["area_ha"].str.replace(",", ".", regex=False), errors="coerce")
for column in ["data_imagem_antes", "data_imagem_depois", "data_publicacao"]:
    df_alerts[column] = pd.to_datetime(df_alerts[column], format="%Y-%m-%d", errors="coerce")
df_alerts["latitude"] = dms_to_decimal(df_alerts["latitude"])
df_alerts["longitude"] = dms_to_decimal(df_alerts["longitude"])

In [ ]:
# Junta as informações da CMF (nem todo alerta da plataforma está na planilha)
df_alerts = df_alerts.merge(df_cmf, on="id_alerta", how="left",
                            validate="one_to_one", indicator="origem")
df_alerts["na_planilha_cmf"] = df_alerts["origem"].eq("both")
df_alerts = df_alerts.drop(columns="origem")
df_alerts["tem_resposta"] = df_alerts["tem_resposta"].fillna(False).astype(bool)

# As duas datas de publicação concordam, onde existem as duas?
comparable = df_alerts["data_publicacao"].notna() & df_alerts["data_publicacao_cmf"].notna()
equal = comparable & df_alerts["data_publicacao"].eq(df_alerts["data_publicacao_cmf"])
print(comparable.sum(), "alertas com as duas datas |", equal.sum(), "iguais")

# Data de publicação: a da plataforma; quando não existe (antes de fev/2025), a da CMF
df_alerts["fonte_data_publicacao"] = np.select(
    [df_alerts["data_publicacao"].notna(), df_alerts["data_publicacao_cmf"].notna()],
    ["plataforma", "planilha CMF"], default="sem data")
df_alerts["data_publicacao"] = df_alerts["data_publicacao"].fillna(df_alerts["data_publicacao_cmf"])
df_alerts = df_alerts.drop(columns="data_publicacao_cmf")
print(df_alerts["fonte_data_publicacao"].value_counts().to_string())

# Município: o da CMF, quando o alerta está na planilha (o Idaf pode ter corrigido após a triagem);
# para os demais, o da plataforma
cmf_city = df_alerts["municipio_cmf"].str.extract(r"^(?P<nome>.+?)\s*-\s*(?P<uf>[A-Z]{2})$")
has_cmf_city = cmf_city["nome"].notna()
print((has_cmf_city & cmf_city["nome"].str.strip().ne(df_alerts["municipio"])).sum(),
      "alertas com município diferente entre CMF e plataforma (vale o da CMF)")
df_alerts.loc[has_cmf_city, "municipio"] = cmf_city.loc[has_cmf_city, "nome"].str.strip()
df_alerts.loc[has_cmf_city, "uf"] = cmf_city.loc[has_cmf_city, "uf"]
df_alerts = df_alerts.drop(columns="municipio_cmf")

In [ ]:
# Checagens bloqueantes
assert df_alerts["id_alerta"].is_unique, "Há alertas repetidos"
assert df_alerts[["municipio", "uf"]].notna().all().all(), "Município fora do padrão 'Nome - UF'"
assert df_alerts["latitude"].dropna().between(-22.0, -17.5).all(), "Latitude fora da região"
assert df_alerts["longitude"].dropna().between(-42.0, -39.0).all(), "Longitude fora da região"
assert df_alerts[["latitude", "longitude", "area_ha"]].notna().all().all(), "Algum valor não converteu"

# Checagens informativas: as duas fontes estão completas uma em relação à outra?
missing_in_sccon = set(df_cmf["id_alerta"]) - set(df_alerts["id_alerta"])
print(len(missing_in_sccon), "alertas da planilha da CMF ausentes no arquivo da SCCON")

is_suppression = df_alerts["tipo_alerta"].str.startswith("Desmatamento")
# Mesma regra do funil (notebook 05): data de publicação; sem ela, data da imagem
date_for_cut = df_alerts["data_publicacao"].fillna(df_alerts["data_imagem_depois"])
before_cut = date_for_cut <= reference_date
not_in_cmf = is_suppression & before_cut & ~df_alerts["na_planilha_cmf"]
print(not_in_cmf.sum(), "alertas de supressão publicados até a data da planilha que não estão na CMF")

In [ ]:
# Salva a tabela na camada prata
silver_dir.mkdir(parents=True, exist_ok=True)
df_alerts.to_parquet(silver_dir / "alertas.parquet", index=False)
print(len(df_alerts), "alertas salvos")
print(df_alerts["tipo_alerta"].value_counts().to_string())